Transformers Would not load for me if I didn't uninstall and reinstall

In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Pip install transformers

In [ ]:
!pip install transformers

In [ ]:
!pip install tick

Loading the model directly using the HF api token.

In [ ]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")

import os
from huggingface_hub import InferenceClient
os.environ["HF_TOKEN"] = ""
client = InferenceClient(
    provider="auto",
    api_key=os.environ["HF_TOKEN"],)

4.1 Sentiment State Extraction API key and Tokenizer

In [ ]:
#Freezing IDFK why

text_example = text_example = [
    "Apple shattered earnings expectations with record revenue",
    "iPhone sales surged beyond analyst forecasts",
    "CEO expressed strong confidence in next quarter guidance",
    "Gross margins expanded significantly year over year",
    "Services revenue hit an all time high",
    "Guidance disappointed investors amid macro uncertainty",
    "Supply chain concerns weighed on the outlook",
    "Management was cautious about consumer spending trends",
    "Operating expenses rose faster than expected",
    "The stock fell after hours on weak forward guidance"
]

results=[]
for text in text_example:
    r = client.text_classification(
        text,
        model="ProsusAI/finbert",
        top_k=3
    )
    results.append(r)

s_i_scores = []
HD_i_vector =[]

# Si(t) is [1,0,-1]*H(Di(t)). Loop keeps both H(Di(t)) and Si(t)
for r in results:
 scores = {d["label"]: d["score"] for d in r}
 positive = scores["positive"]
 neutral = scores["neutral"]
 negative = scores["negative"]

 score = positive-negative
 HD_i_vector.append([positive,neutral,negative])
 s_i_scores.append(score)


print(HD_i_vector,"\n",s_i_scores)

4.1 Using Pipeline

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline
pipe = pipeline("text-classification", model="ProsusAI/finbert", top_k=None)  #top_k=None give pos-nue-neg values


text_example = [
    "Apple shattered earnings expectations with record revenue",
    "iPhone sales surged beyond analyst forecasts",
    "CEO expressed strong confidence in next quarter guidance",
    "Gross margins expanded significantly year over year",
    "Services revenue hit an all time high",
    "Guidance disappointed investors amid macro uncertainty",
    "Supply chain concerns weighed on the outlook",
    "Management was cautious about consumer spending trends",
    "Operating expenses rose faster than expected",
    "The stock fell after hours on weak forward guidance"
]


results = pipe(text_example)


s_i_scores = []
HD_i_vector =[]


# Si(t) is [1,0,-1]*H(Di(t)). Loop keeps both H(Di(t)) and Si(t)
#print(results)
for r in results:
 scores = {d["label"]: d["score"] for d in r}
 positive = scores["positive"]
 neutral = scores["neutral"]
 negative = scores["negative"]


 score = positive-negative
 HD_i_vector.append([positive,neutral,negative])
 s_i_scores.append(score)

print(HD_i_vector,"\n",s_i_scores)

4.2

In [ ]:
import numpy as np

s_off = []
current_sentiment = 0

for sentiment in s_i_scores:
  current_sentiment+=sentiment
  s_off.append(current_sentiment)

'''
s_off[t] is the cumulative sentiment magnitude at the t-th update
If s_i_scores = [.9,-.6,.4] s_off = [.9,.3,.7]
'''
print(s_off)

4.3

In [ ]:
# Section 4.3 - Social Sentiment as a Hawkes Process

import numpy as np
from scipy.optimize import minimize

# -----------------------------------------------------------
# MLE Calibration via scipy
# -----------------------------------------------------------

def hawkes_log_likelihood(params, event_times, T):
    mu, alpha, beta = params

    if mu <= 0 or alpha <= 0 or beta <= 0 or alpha >= beta:
        return np.inf

    log_lik = 0

    for i in range(len(event_times)):
        t = event_times[i]
        excitation = sum(
            alpha * np.exp(-beta * (t - tj))
            for tj in event_times[:i]
        )
        intensity = mu + excitation
        if intensity <= 0:
            return np.inf
        log_lik += np.log(intensity)

    integral = mu * T
    for ti in event_times:
        integral += (alpha / beta) * (1 - np.exp(-beta * (T - ti)))

    log_lik -= integral
    return -log_lik


def calibrate_hawkes(event_times):
    T = event_times[-1]

    best_result = None
    best_val    = np.inf

    initial_guesses = [
        [0.5, 0.3, 1.0],
        [0.2, 0.5, 2.0],
        [0.1, 0.8, 3.0],
        [1.0, 0.5, 2.0],
    ]

    for x0 in initial_guesses:
        result = minimize(
            hawkes_log_likelihood,
            x0=x0,
            args=(event_times, T),
            method='Nelder-Mead',
            options={'maxiter': 10000, 'xatol': 1e-6, 'fatol': 1e-6}
        )
        if result.fun < best_val:
            best_val    = result.fun
            best_result = result

    return best_result.x


# --- Toy timestamps (replace with real tweet timestamps later) ---
# NOTE: one timestamp per text example so indexing stays aligned
tweet_timestamps = np.array([
    0.0, 0.05, 0.06, 0.08,
    0.5, 0.51, 0.53, 0.54, 0.56,
    0.6
])

assert len(tweet_timestamps) == len(s_i_scores), \
    "tweet_timestamps and s_i_scores must have the same length"

mu_soc, alpha, beta = calibrate_hawkes(tweet_timestamps)

print(f"Calibrated mu_soc: {mu_soc:.4f}")
print(f"Calibrated alpha:  {alpha:.4f}")
print(f"Calibrated beta:   {beta:.4f}")
print(f"Stationarity (alpha < beta): {alpha < beta}")

# -----------------------------------------------------------
# Eq. 4: Hawkes conditional intensity at each tweet timestamp
# λ_soc(t) = μ_soc + ∫α*e^(-β(t-u))dN^soc_u
# -----------------------------------------------------------

lambda_soc = []

for i in range(len(tweet_timestamps)):
    t = tweet_timestamps[i]
    excitation = sum(
        alpha * np.exp(-beta * (t - tj))
        for tj in tweet_timestamps[:i]
    )
    intensity = mu_soc + excitation
    lambda_soc.append(intensity)

print("lambda_soc:", lambda_soc)

# -----------------------------------------------------------
# Eq. 5: Volume-weighted moving average sentiment state
# S_soc(t) = ∫s_u dN^soc_u / ∫dN^soc_u
# -----------------------------------------------------------

tau     = 5
epsilon = 1e-6

s_soc = []

for t in range(len(s_i_scores)):
    lower_bound = max(0, t - tau)

    window_scores  = s_i_scores[lower_bound:t+1]
    window_lambdas = lambda_soc[lower_bound:t+1]

    num = sum(s * l for s, l in zip(window_scores, window_lambdas))
    den = sum(window_lambdas) + epsilon

    s_soc.append(num / den)

print("s_soc:", s_soc)

5.1/5.2 (Completely Crutched AI)

In [ ]:
import numpy as np

def compute_volatility(lambda_soc, s_soc, sigma_base=0.2, kappa=0.5):
    sigma_t = []

    for t in range(len(lambda_soc)):
        sigma = sigma_base + kappa * np.log(1 + lambda_soc[t]) * abs(s_soc[t])
        sigma_t.append(sigma)

    return sigma_t

#μ(t,Soff)
def compute_drift(s_off, mu_base=0.01, gamma=0.1):
    mu_t = []

    for t in range(len(s_off)):
        mu = mu_base + gamma * s_off[t]
        mu_t.append(mu)

    return mu_t

#dSt​=St​(μdt+σdW)
def simulate_price(S0, mu_t, sigma_t, dt=1, jump_prob=0.1, mu_J=0, sigma_J=0.05):
    S = [S0]

    for t in range(1, len(mu_t)):
        dW = np.random.normal(0, np.sqrt(dt))

        drift = mu_t[t] * dt
        diffusion = sigma_t[t] * dW

        # Jump component
        jump = 0
        if np.random.rand() < jump_prob:
            J = np.random.normal(mu_J, sigma_J)
            jump = np.exp(J) - 1

        S_new = S[-1] * (1 + drift + diffusion + jump)
        S.append(S_new)

    return S


sigma_t = compute_volatility(lambda_soc, s_soc)

# Step 2: Drift
mu_t = compute_drift(s_off)

# Step 3: Price simulation
S = simulate_price(S0=100, mu_t=mu_t, sigma_t=sigma_t)

S

6.1

In [ ]:
gamma = 0.5  # memory decay parameter

V_soc = []

for t in range(len(s_soc)):
    velocity = 0

    for k in range(t):
        weight = np.exp(-gamma * (t - k))
        dS = s_soc[k+1] - s_soc[k]

        velocity += weight * dS

    V_soc.append(velocity)

print("V_soc:", V_soc)

6.2

In [ ]:
# Section 6.2 - The Divergence Manifold

from scipy.stats import zscore
import numpy as np

# Build delta arrays
delta_P_arr = np.array([0] + [S[t] - S[t-1] for t in range(1, len(S))])
delta_sigma_arr = np.array([0] + [sigma_t[t] - sigma_t[t-1] for t in range(1, len(sigma_t))])
V_soc_arr = np.array(V_soc)

# Normalize each term to the same scale
def safe_zscore(arr):
    std = np.std(arr)
    if std < 1e-8:
        return np.zeros_like(arr, dtype=float)
    return (arr - np.mean(arr)) / std

V_norm = safe_zscore(V_soc_arr)
dP_norm = safe_zscore(delta_P_arr)
ds_norm = safe_zscore(delta_sigma_arr)

# Eq. 9: Z_short(t) = V[S_soc](t) - θ1*ΔPt - θ2*∇σ_IV(t)
theta1 = 0.5
theta2 = 0.5

Z_short = V_norm - theta1 * dP_norm - theta2 * ds_norm

# Definition 6.1: Retail Exuberance Divergence
Gamma_thresh = 0.5

signals = (Z_short > Gamma_thresh).astype(int)

print("Z_short:", Z_short.tolist())
print("Signals:", signals.tolist())

Section 7

In [ ]:
# Section 7 - Neural SDE Architecture

import torch
import torch.nn as nn
import numpy as np

# Section 7.1 - Continuous-Time Neural Networks
# dh_t = f_θ(h_t, S_soc, S_off)dt + g_ϕ(h_t)dW_t

#f_θ
class DriftNetwork(nn.Module):
    """f_θ: models the drift of the latent state h_t"""
    def __init__(self, hidden_dim=32, input_dim=3):
        super().__init__()
        # input: [h_t, S_soc, S_off]
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, h, s_soc, s_off):
        x = torch.cat([h, s_soc, s_off], dim=-1)
        return self.net(x)

#g_ϕ
class DiffusionNetwork(nn.Module):
    """g_ϕ: models the diffusion of the latent state h_t"""
    def __init__(self, hidden_dim=32, input_dim=1):
        super().__init__()
        # input: [h_t] only, per Eq. 10
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
            nn.Softplus()  # ensures diffusion is positive
        )

    def forward(self, h):
        return self.net(h)

#MLP_ψ
class OutputNetwork(nn.Module):
    """MLP_ψ: maps latent state h_t to predicted forward IV (Eq. 11)"""
    def __init__(self, hidden_dim=32, input_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Softplus()  # IV must be positive
        )

    def forward(self, h):
        return self.net(h)


class NeuralSDE(nn.Module):
    """
    Full Neural SDE as proposed in Section 7.
    Integrates the latent state h_t forward using Euler-Maruyama,
    then maps to predicted implied volatility σ_IV(t + Δt).
    """
    def __init__(self, hidden_dim=32, dt=1.0):
        super().__init__()
        self.drift = DriftNetwork(hidden_dim=hidden_dim)
        self.diffusion = DiffusionNetwork(hidden_dim=hidden_dim)
        self.output = OutputNetwork(hidden_dim=hidden_dim)
        self.dt = dt

    def forward(self, s_soc_seq, s_off_seq):
        """
        s_soc_seq: tensor of shape (T,) - social sentiment sequence
        s_off_seq: tensor of shape (T,) - official sentiment sequence
        Returns: predicted IV at each step, shape (T,)
        """
        T = s_soc_seq.shape[0]

        # Initialize latent state h_0
        h = torch.zeros(1, 1)

        iv_preds = []

        for t in range(T):
            s_soc_t = s_soc_seq[t].reshape(1, 1)
            s_off_t = s_off_seq[t].reshape(1, 1)

            # Eq. 10: dh_t = f_θ(h_t, S_soc, S_off)dt + g_ϕ(h_t)dW_t
            f = self.drift(h, s_soc_t, s_off_t)
            g = self.diffusion(h)
            dW = torch.randn(1, 1) * torch.sqrt(torch.tensor(self.dt))

            # Euler-Maruyama integration
            h = h + f * self.dt + g * dW

            # Eq. 11: σ̂_IV(t + Δt) = MLP_ψ(h_t)
            iv_pred = self.output(h)
            iv_preds.append(iv_pred)

        return torch.cat(iv_preds, dim=0).squeeze()


# --- Instantiate and run on toy data ---

# Convert existing variables to tensors
s_soc_tensor = torch.tensor(s_soc, dtype=torch.float32)
s_off_tensor = torch.tensor(s_off, dtype=torch.float32)

# Initialize model
model = NeuralSDE(hidden_dim=32, dt=1.0)

# Forward pass (untrained — random weights, just checking architecture)
with torch.no_grad():
    iv_predictions = model(s_soc_tensor, s_off_tensor)

print("Predicted IV sequence:", iv_predictions.tolist())
print("Shape:", iv_predictions.shape)

Section 8.1 Using text data

In [ ]:
import numpy as np

# =============================================================
# Section 8.1 - Optimal Stopping Problem (Eq. 12)
# U(S,σ,t) = sup E^P[e^{-r(τ-t)}V(σ_IV(τ)) | F_t]
# =============================================================

# --- Option A: Theoretical Placeholder ---
# τ* is the first time Z_short crosses Gamma_thresh
# This is consistent with Definition 6.1


#when does the sentiment divergence signal first fire
def optimal_stopping_placeholder(Z_short, Gamma_thresh=0.5):
    """
    Returns τ* — the first timestep where the divergence
    signal crosses the threshold, i.e. the optimal time
    to enter the short volatility trade.
    """
    for t, z in enumerate(Z_short):
        if z > Gamma_thresh:
            return t
    return None  # no stopping time found in this window

tau_star = optimal_stopping_placeholder(Z_short, Gamma_thresh=0.5)
print("Optimal stopping time τ* (placeholder):", tau_star)

print("Z_short:", Z_short)
print("Signals:", signals)
print("tau_star:", tau_star)

Section 8.1 With real sentiment data

In [ ]:
# --- Option B: Full Numerical Implementation ---
# Solves the value function U(S, σ, t) via dynamic programming
# over a discretized state space.
# Requires real options data in production — uses simulated
# IV surface here for structural correctness.


#when is it mathematically optimal to actually execute the trade?
def compute_straddle_payoff(sigma_IV, S0=100, r=0.05, T=1/252):
    """
    V(σ_IV): theoretical payoff of selling a 0DTE straddle.
    Uses Black-Scholes to price the ATM call + put.
    Selling the straddle = collecting this premium.
    """
    from scipy.stats import norm
    import numpy as np

    if sigma_IV <= 0 or T <= 0:
        return 0.0

    K = S0  # ATM strike
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma_IV**2) * T) / (sigma_IV * np.sqrt(T))
    d2 = d1 - sigma_IV * np.sqrt(T)

    call = S0 * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    put  = K * np.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)

    return call + put  # straddle premium collected


def optimal_stopping_numerical(sigma_t, S, r=0.05, dt=1.0):
    """
    Solves Eq. 12 via backward induction over the simulated
    IV path. At each timestep, compares:
      - stopping now: V(σ_IV(t))
      - continuing:   E[e^{-r*dt} * U(t+1)]
    Returns the value function U and optimal stopping time τ*.

    In production: replace sigma_t with real IV surface data
    and S with real underlying prices.
    """
    T = len(sigma_t)
    discount = np.exp(-r * dt)

    # Initialize value function at terminal time
    U = np.zeros(T)
    U[T-1] = compute_straddle_payoff(sigma_t[T-1], S0=S[T-1])

    # Backward induction
    for t in range(T-2, -1, -1):
        stop_value     = compute_straddle_payoff(sigma_t[t], S0=S[t])
        continue_value = discount * U[t+1]
        U[t]           = max(stop_value, continue_value)

    # τ* is the first time stopping is optimal
    tau_star_numerical = None
    for t in range(T):
        stop_value     = compute_straddle_payoff(sigma_t[t], S0=S[t])
        continue_value = discount * U[t+1] if t+1 < T else 0
        if stop_value >= continue_value:
            tau_star_numerical = t
            break

    return U, tau_star_numerical


U, tau_star_numerical = optimal_stopping_numerical(sigma_t, S)
print("Value function U:", U.tolist())
print("Optimal stopping time τ* (numerical):", tau_star_numerical)

print("Z_short:", Z_short)
print("Signals:", signals)
print("tau_star:", tau_star)

Section 8.2

In [ ]:

# =============================================================
# Section 8.2 - Position Sizing via Fractional Kelly (Eq. 13)
# f*(t) = c * μ_Z(t) / σ²_Z(t)
# =============================================================

def fractional_kelly(Z_short, c=0.5, window=None):
    """
    Computes the optimal fraction of capital f*(t) to deploy
    at each timestep where a signal is active.

    c in (0, 0.5] is the Half-Kelly dampening factor.
    window: optional rolling window for μ_Z and σ²_Z estimation.
            If None, uses all available history up to t.

    Returns f_star: array of capital fractions, 0 if no signal.
    """
    Z = np.array(Z_short)
    f_star = np.zeros(len(Z))

    for t in range(1, len(Z)):
        # Use rolling window or expanding window
        if window:
            start = max(0, t - window)
        else:
            start = 0

        Z_window = Z[start:t]

        if len(Z_window) < 2:
            continue

        mu_Z    = np.mean(Z_window)   # local drift of divergence signal
        sigma2_Z = np.var(Z_window)   # local variance of divergence signal

        if sigma2_Z < 1e-8:
            continue

        f = c * (mu_Z / sigma2_Z)

        # Clip to [0, c] — no leverage, no shorting the signal
        f_star[t] = np.clip(f, 0, c)

    return f_star


f_star = fractional_kelly(Z_short, c=0.5)

print("\nf*(t) capital fractions:", f_star.tolist())

# Show sizing only at signal points
for t, (signal, f) in enumerate(zip(signals, f_star)):
    if signal == 1:
        print(f"  t={t}: SIGNAL ACTIVE — deploy {f*100:.1f}% of capital")
